# 02 — Separação por data: treino, validação e teste

Usa a base limpa gerada no notebook `01_dados_eda.ipynb` (`data/processed/reservas.parquet`): 87.212 reservas, 22 variáveis que passaram na auditoria de vazamento, o alvo `is_canceled` e a data de chegada (só para separar os períodos).

In [1]:
import pandas as pd

df = pd.read_parquet("../data/processed/reservas.parquet")
df.shape

(87212, 24)

## Etapa 3 — Modelagem

### 3.1 Separação por data

O modelo seria usado para prever reservas **futuras**. Então ele treina no passado e é avaliado em meses que vêm depois:
- **Treino:** chegadas de jul/2015 a dez/2016 — onde o modelo aprende.
- **Validação:** jan a abr/2017 — para comparar modelos e ajustar parâmetros.
- **Teste:** mai a ago/2017 — usado **uma vez só**, no fim, para o resultado final. Se o teste fosse usado para escolher o modelo, o resultado final ficaria otimista.

In [2]:
treino = df[df["arrival_date"] < "2017-01-01"]
validacao = df[(df["arrival_date"] >= "2017-01-01") & (df["arrival_date"] < "2017-05-01")]
teste = df[df["arrival_date"] >= "2017-05-01"]

pd.DataFrame({
    "de": [p["arrival_date"].min().date() for p in (treino, validacao, teste)],
    "até": [p["arrival_date"].max().date() for p in (treino, validacao, teste)],
    "reservas": [len(p) for p in (treino, validacao, teste)],
    "% da base": [round(len(p) / len(df), 3) for p in (treino, validacao, teste)],
    "taxa de cancelamento": [round(p["is_canceled"].mean(), 3) for p in (treino, validacao, teste)],
}, index=["treino", "validação", "teste"])

,de,até,reservas,% da base,taxa de cancelamento
treino,2015-07-01,2016-12-31,55580,0.637,0.250
validação,2017-01-01,2017-04-30,13915,0.160,0.284
teste,2017-05-01,2017-08-31,17717,0.203,0.347


A taxa de cancelamento sobe de um período para o outro (25% → 28% → 35%), como visto na EDA. O modelo vai ser avaliado num cenário mais difícil do que o que ele viu no treino — que é o que aconteceria na prática.

Os três períodos ficam salvos em `data/processed/` para os próximos notebooks.

In [3]:
for nome, periodo in [("treino", treino), ("validacao", validacao), ("teste", teste)]:
    periodo.to_parquet(f"../data/processed/{nome}.parquet", index=False)